# Baseline HAP projection model

This model links annual changes in HAP to changes in residential coal and biomass energy shares. Coefficients are designed for use with projected baseline energy consumption.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [2]:
root = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
panel = pd.read_csv(root / "data" / "processed" / "hap_energy_biomass_panel.csv")
panel = panel.query("1990 <= year <= 2019").copy()

## Residential energy shares

Coal and biomass shares are percentages of total residential energy. All remaining fuels form the omitted reference mix.

In [3]:
energy_cols = [c for c in panel if c.startswith("res_") and c.endswith("_ktoe")]
panel["total_res_ktoe"] = panel[energy_cols].sum(axis=1, min_count=1)
panel = panel[panel.total_res_ktoe.gt(0)].copy()

In [4]:
panel["coal_share_pct"] = 100 * panel.res_coa_ktoe / panel.total_res_ktoe
panel["biomass_share_pct"] = 100 * panel.res_bio_ktoe / panel.total_res_ktoe
panel["total_energy_kgoe_pc"] = panel.total_res_ktoe * 1_000_000 / panel.population
panel["ln_total_energy_pc"] = np.log(panel.total_energy_kgoe_pc)
panel["hap_solid_pct"] = 100 * panel.hap_solid_fuels

In [5]:
model_cols = ["hap_solid_pct", "coal_share_pct", "biomass_share_pct", "ln_total_energy_pc"]
model_panel = panel.dropna(subset=model_cols).sort_values(["iso3", "year"]).copy()
pd.Series({"observations": len(model_panel), "countries": model_panel.iso3.nunique()})

observations    4102
countries        139
dtype: int64

## Change model

The coefficients measure the change in HAP percentage points associated with a one-percentage-point change in each fuel share. Standard errors are clustered by country.

In [6]:
changes = model_panel.groupby("iso3")[model_cols].diff().add_prefix("d_")
change_panel = pd.concat([model_panel[["iso3", "year"]], changes], axis=1)
change_panel["year_gap"] = model_panel.groupby("iso3").year.diff()
change_panel = change_panel.query("year_gap == 1").dropna()

In [7]:
change_model = smf.ols(
    "d_hap_solid_pct ~ d_coal_share_pct + d_biomass_share_pct + d_ln_total_energy_pc",
    data=change_panel
).fit(cov_type="cluster", cov_kwds={"groups": change_panel.iso3})

In [8]:
terms = ["Intercept", "d_coal_share_pct", "d_biomass_share_pct", "d_ln_total_energy_pc"]
results = pd.DataFrame({
    "coefficient": change_model.params[terms], "std_error": change_model.bse[terms],
    "p_value": change_model.pvalues[terms],
    "ci_lower": change_model.conf_int().loc[terms, 0], "ci_upper": change_model.conf_int().loc[terms, 1]
})
results.round(4)

,coefficient,std_error,p_value,ci_lower,ci_upper
Intercept,-0.5207,0.0431,0.0000,-0.6052,-0.4363
d_coal_share_pct,0.0043,0.0089,0.6315,-0.0132,0.0217
d_biomass_share_pct,0.0346,0.0084,0.0000,0.0182,0.0510
d_ln_total_energy_pc,-0.1602,0.1204,0.1835,-0.3962,0.0759


## Coal robustness checks

The checks use five-year changes, countries with at least a 1% coal share, and an approximate coal-plus-charcoal measure. The last check adds UN charcoal to the energy total because the two sources do not identify overlap consistently.

In [9]:
def fit_change(data, outcome, predictors, lag=1):
    cols = [outcome] + predictors
    diff = data[["iso3", "year"]].join(data.groupby("iso3")[cols].diff(lag).add_prefix("d_"))
    diff["year_gap"] = data.groupby("iso3").year.diff(lag)
    diff = diff.query("year_gap == @lag").dropna()
    formula = f"d_{outcome} ~ " + " + ".join(f"d_{x}" for x in predictors)
    model = smf.ols(formula, diff).fit(cov_type="cluster", cov_kwds={"groups": diff.iso3})
    return model

In [10]:
five_year_model = fit_change(
    model_panel, "hap_solid_pct",
    ["coal_share_pct", "biomass_share_pct", "ln_total_energy_pc"], lag=5
)
active_iso3 = model_panel.groupby("iso3").coal_share_pct.max().loc[lambda x: x >= 1].index
active_coal_model = fit_change(
    model_panel[model_panel.iso3.isin(active_iso3)], "hap_solid_pct",
    ["coal_share_pct", "biomass_share_pct", "ln_total_energy_pc"]
)

In [11]:
aligned = model_panel.dropna(subset=["hh_charcoal_ktoe"]).copy()
aligned["aligned_total_ktoe"] = aligned.total_res_ktoe + aligned.hh_charcoal_ktoe
aligned["coal_charcoal_share_pct"] = 100 * (aligned.res_coa_ktoe + aligned.hh_charcoal_ktoe) / aligned.aligned_total_ktoe
aligned["ln_aligned_energy_pc"] = np.log(aligned.aligned_total_ktoe * 1_000_000 / aligned.population)
aligned["hap_coal_charcoal_pct"] = 100 * aligned.hap_coal_charcoal
aligned_model = fit_change(
    aligned, "hap_coal_charcoal_pct", ["coal_charcoal_share_pct", "ln_aligned_energy_pc"]
)

In [12]:
checks = [
    ("Main: coal", change_model, "d_coal_share_pct"),
    ("Five-year: coal", five_year_model, "d_coal_share_pct"),
    ("Coal-use countries", active_coal_model, "d_coal_share_pct"),
    ("Coal + charcoal aligned", aligned_model, "d_coal_charcoal_share_pct"),
    ("Five-year: biomass", five_year_model, "d_biomass_share_pct")
]
pd.DataFrame({
    "coefficient": [m.params[t] for _, m, t in checks],
    "std_error": [m.bse[t] for _, m, t in checks],
    "p_value": [m.pvalues[t] for _, m, t in checks],
    "observations": [int(m.nobs) for _, m, _ in checks]
}, index=[name for name, _, _ in checks]).round(4)

,coefficient,std_error,p_value,observations
Main: coal,0.0043,0.0089,0.6315,3963
Five-year: coal,-0.0118,0.0370,0.7491,3407
Coal-use countries,-0.0003,0.0075,0.9634,1345
Coal + charcoal aligned,0.0019,0.0066,0.7718,2229
Five-year: biomass,0.1238,0.0281,0.0000,3407


The intercept is the estimated annual background change in HAP after accounting for energy variables. It provides the common trend used in baseline projections.

## Projection function

The function anchors projections to the latest observed HAP for each country. A scenario must contain `iso3`, `year`, `coal_share_pct`, `biomass_share_pct`, and `total_energy_kgoe_pc`.

In [13]:
def project_baseline(base, scenario, model=change_model):
    base_cols = ["iso3", "year", "hap_solid_pct", "coal_share_pct", "biomass_share_pct", "ln_total_energy_pc"]
    base = base[base_cols].rename(columns={c: f"base_{c}" for c in base_cols if c != "iso3"})
    out = scenario.merge(base, on="iso3", validate="many_to_one").copy()
    out["ln_total_energy_pc"] = np.log(out.total_energy_kgoe_pc)
    b = model.params
    out["hap_baseline_pct"] = out.base_hap_solid_pct + b.Intercept * (out.year - out.base_year)
    out["hap_baseline_pct"] += b.d_coal_share_pct * (out.coal_share_pct - out.base_coal_share_pct)
    out["hap_baseline_pct"] += b.d_biomass_share_pct * (out.biomass_share_pct - out.base_biomass_share_pct)
    out["hap_baseline_pct"] += b.d_ln_total_energy_pc * (out.ln_total_energy_pc - out.base_ln_total_energy_pc)
    out["hap_baseline_pct"] = out.hap_baseline_pct.clip(0, 100)
    return out

## Historical backtest

The model is anchored in 2010 and uses observed energy changes to predict HAP through 2019.

In [14]:
base_2010 = model_panel.query("year == 2010")
scenario_cols = ["iso3", "year", "coal_share_pct", "biomass_share_pct", "total_energy_kgoe_pc"]
observed_scenario = model_panel.query("2011 <= year <= 2019")[scenario_cols]
backtest = project_baseline(base_2010, observed_scenario)
backtest = backtest.merge(model_panel[["iso3", "year", "hap_solid_pct"]], on=["iso3", "year"])

In [15]:
error = backtest.hap_baseline_pct - backtest.hap_solid_pct
pd.Series({
    "backtest observations": len(backtest),
    "mean absolute error (pp)": error.abs().mean(),
    "root mean squared error (pp)": np.sqrt(np.mean(error**2)),
    "mean error (pp)": error.mean()
}).round(3)

backtest observations           1242.000
mean absolute error (pp)           1.978
root mean squared error (pp)       3.546
mean error (pp)                    0.653
dtype: float64

The model is intended for baseline scenario analysis, not causal attribution. Future work should add projected income, urbanization, electrification, and uncertainty intervals around scenario results.